In [0]:
# =========================================================
# SILVER LAYER: SETUP, PARAMETERS, AND AUDIT
# =========================================================

from datetime import datetime, timezone
import uuid

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, LongType
)
from delta.tables import DeltaTable

# Widgets are recreated so rerunning this setup cell is predictable.
dbutils.widgets.removeAll()
dbutils.widgets.dropdown("load_type", "full", ["full", "incremental", "backfill"])
dbutils.widgets.text("start_date", "2011-01-01")
dbutils.widgets.text("end_date", "2026-09-30")
dbutils.widgets.text("batch_id", "")

load_type = dbutils.widgets.get("load_type").strip().lower()
start_date = dbutils.widgets.get("start_date").strip()
end_date = dbutils.widgets.get("end_date").strip()
batch_id = dbutils.widgets.get("batch_id").strip()

bronze_table = "bronze.world_bank_food_prices"
silver_table = "silver.world_bank_food_prices"
reject_table = "rejects.silver_food_price_rejects_v2"
schema_drift_table = "rejects.silver_schema_drift_logs"
audit_table = "audit.pipeline_runs"
source_name = "World Bank Real-Time Food Prices"
run_id = str(uuid.uuid4())
pipeline_start = datetime.now(timezone.utc).replace(tzinfo=None)

records_read = 0
records_inserted = 0
records_updated = 0
records_rejected = 0

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS rejects")
spark.sql("CREATE SCHEMA IF NOT EXISTS audit")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {audit_table} (
    run_id STRING,
    source_name STRING,
    layer STRING,
    load_type STRING,
    parameter_processed STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    records_read BIGINT,
    records_inserted BIGINT,
    records_updated BIGINT,
    records_rejected BIGINT,
    status STRING,
    error_message STRING
) USING DELTA
""")
if "parameter_processed" not in spark.table(audit_table).columns:
    spark.sql(f"ALTER TABLE {audit_table} ADD COLUMNS (parameter_processed STRING)")

AUDIT_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("source_name", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("parameter_processed", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("records_read", LongType(), True),
    StructField("records_inserted", LongType(), True),
    StructField("records_updated", LongType(), True),
    StructField("records_rejected", LongType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True),
])

def write_audit(status, error_message=None):
    """Append one Silver run status row; columns are aligned to the existing table."""
    end_time = datetime.now(timezone.utc).replace(tzinfo=None)
    parameter_processed = (
        f"load_type={load_type}; start_date={start_date}; "
        f"end_date={end_date}; batch_id={batch_id or '(none)'}"
    )
    message = str(error_message)[:4000] if error_message is not None else None
    row = spark.createDataFrame([(
        run_id, source_name, "Silver", load_type, parameter_processed,
        pipeline_start, end_time, int(records_read), int(records_inserted),
        int(records_updated), int(records_rejected), status, message
    )], schema=AUDIT_SCHEMA)
    target_columns = spark.table(audit_table).columns
    row.select(*target_columns).write.format("delta").mode("append").saveAsTable(audit_table)

print("Silver run ID:", run_id)
print("Load type:", load_type)
print("Date range:", start_date, "to", end_date)
print("Batch ID:", batch_id or "(not specified)")


In [0]:
# =========================================================
# READ AND FILTER BRONZE SOURCE
# =========================================================

try:
    from datetime import datetime as _dt
    if load_type not in ("full", "incremental", "backfill"):
        raise ValueError("load_type must be full, incremental, or backfill")
    start_dt = _dt.strptime(start_date, "%Y-%m-%d").date()
    end_dt = _dt.strptime(end_date, "%Y-%m-%d").date()
    if start_dt > end_dt:
        raise ValueError("start_date must be on or before end_date")

    if not spark.catalog.tableExists(bronze_table):
        raise ValueError(f"Bronze source table does not exist: {bronze_table}")

    bronze_df = spark.table(bronze_table)
    required_columns = ["ISO3", "country", "geo_id", "price_date", "lat", "lon"]
    missing_columns = [c for c in required_columns if c not in bronze_df.columns]
    if missing_columns:
        raise ValueError(f"Bronze table is missing required columns: {missing_columns}")

    if load_type in ("incremental", "backfill"):
        bronze_df = bronze_df.filter(
            (F.to_date(F.col("price_date")) >= F.lit(start_dt)) &
            (F.to_date(F.col("price_date")) <= F.lit(end_dt))
        )

    if batch_id:
        if "_batch_id" in bronze_df.columns:
            bronze_df = bronze_df.filter(F.col("_batch_id") == batch_id)
        elif "_run_id" in bronze_df.columns:
            # Current Bronze metadata has _run_id, not necessarily _batch_id.
            # If entering a batch_id while _batch_id is absent, this treats it as a Bronze run_id.
            bronze_df = bronze_df.filter(F.col("_run_id") == batch_id)
        else:
            raise ValueError("batch_id was supplied, but Bronze has neither _batch_id nor _run_id")

    records_read = bronze_df.count()
    if records_read == 0:
        raise ValueError("The selected parameters returned zero Bronze rows; no Silver write was attempted.")

    print("Bronze rows selected for Silver:", records_read)
    print("Source columns:", len(bronze_df.columns))
    bronze_df.printSchema()

except Exception as exc:
    write_audit("FAILURE", exc)
    raise


In [0]:
# =========================================================
# CLEAN AND CAST SILVER COLUMNS 
# =========================================================

try:
    staging_table = "silver.world_bank_food_prices_staging"

    source_columns = bronze_df.columns
    string_columns = {
        "ISO3", "country", "adm1_name", "adm2_name", "mkt_name", "geo_id",
        "currency", "components", "start_dense_data", "last_survey_point",
        "price_type", "commodity", "unit", "_run_id", "_batch_id",
        "_source_system", "_source_file"
    }
    date_columns = {"price_date"}
    integer_columns = {"year", "month", "mp_year", "mp_month"}
    boolean_columns = {"spatially_interpolated"}

    def clean(c):
        return f"NULLIF(TRIM(`{c}`), '')"      # trim, and turn empty text into NULL

    numeric_columns = [
        c for c in source_columns
        if c not in string_columns | date_columns | integer_columns | boolean_columns
        | {"_ingestion_timestamp", "load_timestamp"}
    ]

    # Build every column expression in a list and apply it with ONE select.
    select_list = []
    for c in source_columns:
        if c == "load_timestamp":
            continue                              # replaced below with the Silver load time
        if c == "_ingestion_timestamp":
            select_list.append(F.col(c))
        elif c in date_columns:
            select_list.append(F.expr(f"try_cast({clean(c)} AS DATE)").alias(c))
        elif c in integer_columns:
            select_list.append(F.expr(f"try_cast({clean(c)} AS INT)").alias(c))
        elif c in boolean_columns:
            select_list.append(F.expr(f"try_cast({clean(c)} AS BOOLEAN)").alias(c))
        elif c in string_columns:
            expr = f"UPPER({clean(c)})" if c == "ISO3" else clean(c)
            select_list.append(F.expr(expr).alias(c))
        else:
            select_list.append(F.expr(f"try_cast({clean(c)} AS DOUBLE)").alias(c))

    typed_df = bronze_df.select(
        *select_list,
        F.current_timestamp().alias("load_timestamp"),
        F.lit(run_id).alias("_silver_run_id"),
        F.current_timestamp().alias("_silver_processed_at"),
    )

    # Save once. Every later step reads this saved table instead of recomputing.
    (typed_df.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(staging_table))
    staged = spark.table(staging_table)

    # Report values that could not be cast. They become NULL; the run does not crash.
    check_columns = numeric_columns + [c for c in (integer_columns | boolean_columns) if c in source_columns]
    raw_counts = bronze_df.agg(*[
        F.count(F.when(F.trim(F.col(f"`{c}`")) != "", 1)).alias(c) for c in check_columns]).first().asDict()
    typed_counts = staged.agg(*[
        F.count(F.col(f"`{c}`")).alias(c) for c in check_columns]).first().asDict()
    lost = [(run_id, c, int(raw_counts[c] - typed_counts[c]))
            for c in check_columns if raw_counts[c] > typed_counts[c]]
    if lost:
        (spark.createDataFrame(lost, "run_id string, column_name string, bad_value_count long")
              .withColumn("load_timestamp", F.current_timestamp())
              .write.format("delta").mode("append").option("mergeSchema", "true")
              .saveAsTable("rejects.silver_type_violations"))

    print("Columns cast:", len(check_columns))
    print("Columns with values that could not be cast:", len(lost))

except Exception as exc:
    write_audit("FAILURE", exc)
    raise

In [0]:
# =========================================================
# VALIDATE AND QUARANTINE INVALID SILVER ROWS
# =========================================================

try:
    staged_df = spark.table(staging_table)

    errors = [
        F.when(F.col("ISO3").isNull() | (F.length(F.col("ISO3")) != 3), F.lit("Missing or invalid ISO3")),
        F.when(F.col("country").isNull(), F.lit("Missing country")),
        F.when(F.col("geo_id").isNull(), F.lit("Missing geo_id")),
        F.when(F.col("price_date").isNull(), F.lit("Missing or invalid price_date")),
        F.when(F.col("lat").isNull() | (F.col("lat") < -90) | (F.col("lat") > 90), F.lit("Invalid latitude")),
        F.when(F.col("lon").isNull() | (F.col("lon") < -180) | (F.col("lon") > 180), F.lit("Invalid longitude")),
        F.when(F.col("month").isNotNull() & ((F.col("month") < 1) | (F.col("month") > 12)), F.lit("Invalid month")),
    ]
    staged_df = staged_df.withColumn(
        "_validation_error", F.concat_ws("; ", *[e.otherwise(None) for e in errors]))

    invalid_df = staged_df.filter(F.col("_validation_error") != "")
    valid_df = staged_df.filter(F.col("_validation_error") == "").drop("_validation_error")

    records_rejected = invalid_df.count()
    valid_count = valid_df.count()
    print("Valid Silver candidates:", valid_count)
    print("Rejected Silver records:", records_rejected)

    if records_rejected > 0:
        (invalid_df.select(
            "ISO3", "country", "geo_id", "price_date", "lat", "lon", "year", "month",
            F.col("_validation_error").alias("reject_reason"),
            F.lit(run_id).alias("reject_run_id"),
            F.current_timestamp().alias("rejected_at"),
            "load_timestamp")
         .write.format("delta").option("mergeSchema", "true").mode("append")
         .saveAsTable(reject_table))
        print("Rejected rows appended to:", reject_table)

    if valid_count == 0:
        raise ValueError("All selected source rows were rejected; Silver target was not changed.")

except Exception as exc:
    write_audit("FAILURE", exc)
    raise

In [0]:
# =========================================================
# DEDUPLICATE AND MERGE INTO SILVER
# =========================================================

try:
    key_columns = ["ISO3", "geo_id", "price_date"]

    before_deduplication = valid_df.count()
    deduplicated_count = valid_df.select(*key_columns).distinct().count()

    if deduplicated_count == before_deduplication:
        deduplicated_df = valid_df          # no duplicates, so no window step needed
    else:
        window_spec = Window.partitionBy(*key_columns).orderBy(
            F.col("_ingestion_timestamp").desc_nulls_last(),
            F.col("_run_id").desc_nulls_last())
        deduplicated_df = (
            valid_df.withColumn("_duplicate_rank", F.row_number().over(window_spec))
            .filter(F.col("_duplicate_rank") == 1)
            .drop("_duplicate_rank"))

    print("Rows before deduplication:", before_deduplication)
    print("Rows after deduplication:", deduplicated_count)
    print("Duplicates removed:", before_deduplication - deduplicated_count)

    if deduplicated_count == 0:
        raise ValueError("No valid deduplicated records remain; Silver was not changed.")
    # Create the target the first time. Existing target schemas must be compatible.
    if not spark.catalog.tableExists(silver_table):
        (deduplicated_df.write.format("delta").saveAsTable(silver_table))
        records_inserted = deduplicated_count
        records_updated = 0
        print("Created Silver table:", silver_table)
    else:
        target_fields = {f.name: f.dataType for f in spark.table(silver_table).schema.fields}
        source_fields = {f.name: f.dataType for f in deduplicated_df.schema.fields}

        new_columns = [c for c in source_fields if c not in target_fields]
        missing_source_columns = [c for c in target_fields if c not in source_fields]
        type_mismatches = [
            (c, source_fields[c].simpleString(), target_fields[c].simpleString())
            for c in source_fields.keys() & target_fields.keys()
            if source_fields[c] != target_fields[c]
        ]

        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {schema_drift_table} (
            run_id STRING,
            column_name STRING,
            drift_type STRING,
            source_type STRING,
            target_type STRING,
            detected_at TIMESTAMP
        ) USING DELTA
        """)

        drift_rows = []
        detected_at = datetime.now(timezone.utc).replace(tzinfo=None)
        for c in new_columns:
            drift_rows.append((run_id, c, "NEW_SOURCE_COLUMN", source_fields[c].simpleString(), None, detected_at))
        for c in missing_source_columns:
            drift_rows.append((run_id, c, "MISSING_FROM_SOURCE", None, target_fields[c].simpleString(), detected_at))
        for c, source_type, target_type in type_mismatches:
            drift_rows.append((run_id, c, "TYPE_MISMATCH", source_type, target_type, detected_at))
        if drift_rows:
            drift_schema = "run_id string, column_name string, drift_type string, source_type string, target_type string, detected_at timestamp"
            spark.createDataFrame(drift_rows, drift_schema).write.format("delta").mode("append").saveAsTable(schema_drift_table)

        # Add new source columns to the target so compatible schema additions can continue.
        for c in new_columns:
            sql_type = source_fields[c].simpleString().upper()
            spark.sql(f"ALTER TABLE {silver_table} ADD COLUMNS (`{c}` {sql_type})")

        if type_mismatches:
            details = ", ".join([f"{c}: source={s}, target={t}" for c, s, t in type_mismatches[:20]])
            raise ValueError(
                "Existing Silver columns have incompatible data types: " + details + ". "
                "If this is the first run after changing the Silver type rules, back up and drop/rebuild "
                "the derived Silver table once. Do not drop Bronze. Drift details were logged."
            )

        # For source columns missing in this batch, supply typed NULLs to preserve target schema.
        for c in missing_source_columns:
            deduplicated_df = deduplicated_df.withColumn(c, F.lit(None).cast(target_fields[c]))

        target_columns = spark.table(silver_table).columns
        deduplicated_df = deduplicated_df.select(*target_columns)

        target = DeltaTable.forName(spark, silver_table)
        merge_condition = " AND ".join([f"t.`{c}` = s.`{c}`" for c in key_columns])
        metadata_columns = set(key_columns + [
            "load_timestamp", "_silver_run_id", "_silver_processed_at",
            "_ingestion_timestamp", "_run_id", "_batch_id",
            "_source_system", "_source_file"
        ])
        # If a source column is absent in this batch, preserve the target's existing
        # value on matched rows instead of overwriting it with the typed NULL added above.
        metadata_columns.update(missing_source_columns)
        business_columns = [c for c in deduplicated_df.columns if c not in metadata_columns]
        changed_condition = " OR ".join([f"NOT (t.`{c}` <=> s.`{c}`)" for c in business_columns])
        update_values = {
            c: f"s.`{c}`" for c in deduplicated_df.columns
            if c not in missing_source_columns
        }
        insert_values = {c: f"s.`{c}`" for c in deduplicated_df.columns}

        (target.alias("t")
            .merge(deduplicated_df.alias("s"), merge_condition)
            .whenMatchedUpdate(condition=changed_condition, set=update_values)
            .whenNotMatchedInsert(values=insert_values)
            .execute())

        # Read true row-action metrics from the latest Delta commit.
        operation_metrics = DeltaTable.forName(spark, silver_table).history(1).select("operationMetrics").first()[0] or {}
        records_inserted = int(operation_metrics.get("numTargetRowsInserted", operation_metrics.get("numInsertedRows", "0")))
        records_updated = int(operation_metrics.get("numTargetRowsUpdated", operation_metrics.get("numUpdatedRows", "0")))
        print("Delta MERGE inserted:", records_inserted)
        print("Delta MERGE updated:", records_updated)

    # Post-write assertions.
    silver_check = spark.table(silver_table)
    duplicate_groups = (silver_check.groupBy(*key_columns).count().filter(F.col("count") > 1).count())
    null_keys = silver_check.filter(
        F.col("ISO3").isNull() | F.col("geo_id").isNull() | F.col("price_date").isNull()
    ).count()
    missing_load_timestamps = silver_check.filter(F.col("load_timestamp").isNull()).count()
    if duplicate_groups or null_keys or missing_load_timestamps:
        raise ValueError(
            f"Silver quality check failed: duplicate_groups={duplicate_groups}, "
            f"null_keys={null_keys}, missing_load_timestamps={missing_load_timestamps}"
        )

    print("Silver row count:", silver_check.count())
    print("Silver business-key and load_timestamp checks passed.")
    write_audit("SUCCESS")

except Exception as exc:
    write_audit("FAILURE", exc)
    raise


In [0]:
# =========================================================
# SILVER POST-RUN DIAGNOSTICS
# =========================================================

spark.sql("DROP TABLE IF EXISTS silver.world_bank_food_prices_staging")   # temporary table

silver_check = spark.table(silver_table)
print("Silver rows:", silver_check.count())
print("Silver columns:", len(silver_check.columns))
silver_check.printSchema()

display(silver_check.select(
    "ISO3", "country", "geo_id", "price_date", "year", "month", "lat", "lon", "load_timestamp"
).limit(20))

display(spark.sql(f"""
    SELECT run_id, load_type, parameter_processed, start_time, end_time,
           records_read, records_inserted, records_updated, records_rejected,
           status, error_message
    FROM {audit_table}
    WHERE layer = 'Silver'
    ORDER BY start_time DESC
    LIMIT 10
"""))


In [0]:
tables_to_check = [
    "bronze.world_bank_food_prices",
    "silver.world_bank_food_prices",
    "rejects.silver_food_price_rejects_v2",
    "audit.pipeline_runs"
]

for table_name in tables_to_check:
    try:
        if spark.catalog.tableExists(table_name):
            print(f"{table_name}: {spark.table(table_name).count()} rows")
        else:
            print(f"{table_name}: TABLE DOES NOT EXIST")
    except Exception as e:
        print(f"{table_name}: ERROR - {e}")

In [0]:
spark.sql("""
    SELECT
        run_id,
        layer,
        load_type,
        records_read,
        records_inserted,
        records_updated,
        records_rejected,
        status,
        error_message,
        start_time,
        end_time
    FROM audit.pipeline_runs
    ORDER BY start_time DESC
    LIMIT 10
""").show(truncate=False)

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("silver.world_bank_food_prices")

duplicate_keys = (
    silver_df
    .groupBy("ISO3", "geo_id", "price_date")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate business-key groups:", duplicate_keys.count())
display(duplicate_keys.limit(20))

In [0]:
silver_df = spark.table("silver.world_bank_food_prices")

silver_df.select(
    F.sum(F.col("ISO3").isNull().cast("int")).alias("missing_ISO3"),
    F.sum(F.col("geo_id").isNull().cast("int")).alias("missing_geo_id"),
    F.sum(F.col("price_date").isNull().cast("int")).alias("missing_price_date"),
    F.sum(F.col("load_timestamp").isNull().cast("int")).alias("missing_load_timestamp")
).show()

In [0]:
rejects_df = spark.table("rejects.silver_food_price_rejects_v2")

print("Total rejected records:", rejects_df.count())

rejects_df.groupBy("reject_reason") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(20, truncate=False)

In [0]:
silver_df = spark.table("silver.world_bank_food_prices")

silver_df.printSchema()

In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table("bronze.world_bank_food_prices")
rejects_df = spark.table("rejects.silver_food_price_rejects_v2")

print("Bronze coordinate summary:")
bronze_df.select(
    F.count("*").alias("total"),
    F.sum(F.col("lat").isNull().cast("int")).alias("latitude_null"),
    F.sum(F.col("lon").isNull().cast("int")).alias("longitude_null")
).show()

print("Sample rejected records:")
display(
    rejects_df.select(
        "ISO3",
        "geo_id",
        "price_date",
        "lat",
        "lon",
        "reject_reason"
    ).limit(20)
)

In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table("bronze.world_bank_food_prices")

bronze_df.select(
    "ISO3",
    "geo_id",
    "price_date",
    "lat",
    "lon"
).show(10, truncate=False)

print("Bronze data types:")
print("lat:", bronze_df.schema["lat"].dataType)
print("lon:", bronze_df.schema["lon"].dataType)

In [0]:
staging_table = "silver.world_bank_food_prices_staging"

if spark.catalog.tableExists(staging_table):
    staging_df = spark.table(staging_table)

    print("Staging coordinate types:")
    print("lat:", staging_df.schema["lat"].dataType)
    print("lon:", staging_df.schema["lon"].dataType)

    staging_df.select(
        "ISO3",
        "geo_id",
        "price_date",
        "lat",
        "lon"
    ).show(10, truncate=False)
else:
    print("Staging table does not exist.")

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("silver.world_bank_food_prices")

silver_df.select(
    F.count("*").alias("total_silver_rows"),

    F.sum(
        F.col("lat").isNull().cast("int")
    ).alias("null_lat"),

    F.sum(
        F.col("lon").isNull().cast("int")
    ).alias("null_lon"),

    F.sum(
        ((F.col("lat") < -90) | (F.col("lat") > 90))
        .cast("int")
    ).alias("invalid_lat"),

    F.sum(
        ((F.col("lon") < -180) | (F.col("lon") > 180))
        .cast("int")
    ).alias("invalid_lon")
).show()

In [0]:
bronze_df = spark.table("bronze.world_bank_food_prices")
rejects_df = spark.table("rejects.silver_food_price_rejects_v2")

bronze_df.alias("b").join(
    rejects_df.select(
        "ISO3", "geo_id", "price_date"
    ).alias("r"),
    on=["ISO3", "geo_id", "price_date"],
    how="inner"
).select(
    "ISO3", "geo_id", "price_date",
    F.col("b.lat").alias("bronze_lat"),
    F.col("b.lon").alias("bronze_lon")
).show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table("bronze.world_bank_food_prices")

bronze_df.select(
    F.sum(
        (
            F.col("lat").isNull() |
            (F.trim(F.col("lat")) == "")
        ).cast("int")
    ).alias("missing_or_blank_lat"),

    F.sum(
        (
            F.col("lon").isNull() |
            (F.trim(F.col("lon")) == "")
        ).cast("int")
    ).alias("missing_or_blank_lon"),

    F.sum(
        (
            F.col("lat").isNotNull() &
            (F.trim(F.col("lat")) != "") &
            F.col("lat").cast("double").isNull()
        ).cast("int")
    ).alias("unparseable_lat"),

    F.sum(
        (
            F.col("lon").isNotNull() &
            (F.trim(F.col("lon")) != "") &
            F.col("lon").cast("double").isNull()
        ).cast("int")
    ).alias("unparseable_lon")
).show()

In [0]:
bronze_df.alias("b").join(
    spark.table("rejects.silver_food_price_rejects_v2")
    .select("ISO3", "geo_id", "price_date")
    .alias("r"),
    ["ISO3", "geo_id", "price_date"],
    "inner"
).select(
    "ISO3",
    "geo_id",
    "price_date",
    F.col("b.lat").alias("original_lat"),
    F.col("b.lon").alias("original_lon")
).distinct().show(30, truncate=False)